# Multi-Disease Medical AI — Module 1: Chest X-Ray Model Training & Grad-CAM
### ECE Major Project
**Modality:** Chest Radiography (ChestMNIST / NIH ChestX-ray14 subset, $224 \times 224$)
**Objective:** Multi-label classification across 14 thoracic pathologies + Grad-CAM Explainable AI.

In [ ]:
!pip install -q medmnist torchvision scikit-learn opencv-python-headless matplotlib tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.models as tv_models
from medmnist import ChestMNIST
from sklearn.metrics import roc_auc_score
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Compute Device:", device)

In [ ]:
# Data Transforms with DIP Normalization
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) if False else transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x)
])

train_dataset = ChestMNIST(split="train", download=True, size=224, transform=transform)
val_dataset   = ChestMNIST(split="val", download=True, size=224, transform=transform)
test_dataset  = ChestMNIST(split="test", download=True, size=224, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Batches in Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

## Build SOTA DenseNet-121 (CheXNet Architecture)

In [ ]:
model = tv_models.densenet121(weights=tv_models.DenseNet121_Weights.DEFAULT)
num_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, 14)  # 14 thoracic disease labels
)
model = model.to(device)

# Binary Cross Entropy for Multi-Label Classification
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

In [ ]:
# Training & Validation Routine
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for images, targets in tqdm(loader, desc="Training"):
        images = images.to(device).float()
        targets = targets.to(device).float()
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device).float()
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            all_preds.append(probs.cpu().numpy())
            all_targets.append(targets.numpy())
    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    # Compute Macro ROC-AUC across all 14 labels
    aucs = [roc_auc_score(all_targets[:, i], all_preds[:, i]) for i in range(14) if len(np.unique(all_targets[:, i])) > 1]
    return float(np.mean(aucs)), all_preds, all_targets

## Grad-CAM Explainable AI on Chest X-Rays
Highlighting regions indicating Atelectasis, Cardiomegaly, Pneumonia, or Effusion.

In [ ]:
# Grad-CAM Hook Demonstration
target_layer = model.features.denseblock4.denselayer16.conv2
print("Grad-CAM Target Layer configured:", target_layer)
# Model weights can be saved for the Streamlit dashboard:
torch.save(model.state_dict(), "chest_densenet121.pth")
print("Model checkpoint saved successfully.")